# ESEEM: an electron spin-1/2 coupled to a nuclear spin-1/2

This notebook simulates Hahn echo and three-pulse electron spin echo envelope modulation
(ESEEM) experiments on a electron spin-1/2 which is hyperfine coupled to a nuclear spin-1/2, e.g. $^{13}\mathrm{C}$.  
The notebook covers: 
- Spin Hamiltonian with hyperfine coupling
- Analytical envelopes
- Hahn echo 
- three-pulse ESEEM 
- thermal initialization
- weaker Rabi drive
- effect of relaxation and dephasing of the electron spin


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import Qobj, expect

import spinfinity
from spinfinity import CC, SpinSys, hamil, bath, lind
from spinfinity import helperfunctions as hf

## Static spin Hamiltonian

Spin 0 is the electron ($S=1/2$); spin 1 is the nucleus ($I=1/2$).

$$
H=-g_e\mu_B\mathbf B_0\cdot\mathbf S
  -g_{\mathrm{n,eff}}\mu_B\mathbf B_0\cdot\mathbf I
  +A_\perp(S_xI_x+S_yI_y)+A_\parallel S_zI_z
$$

THe example uses $B_0=0.5 ~\mathrm{T}$, tilted $45^\circ$ from the hyperfine $z$ axis, and
$(A_\perp,A_\parallel)/h=(2,8) ~\mathrm{MHz}$. The nuclear ordinary gyromagnetic
ratio is approximately $10.7084 ~\mathrm{MHz/T}$ Spinfinity uses the Bohr magneton
for both spins and converts this ratio into an effective nuclear g-factor.


In [ ]:
spin = SpinSys([0.5, 0.5])
spin.ReadoutSpin = 0  # Performing the measurement on the electron spin

# Magnetic field
B0 = 0.5  # T
theta = np.deg2rad(45.0)
n = np.array([np.sin(theta), 0.0, np.cos(theta)])
spin.B = B0 * n
spin.BTip = [0.0, 0.0, 0.0]

# g-factors
ge = 2.0023
gamma_n = 10.7084e-3  # ordinary nuclear gyromagnetic ratio, GHz/T
g_n_eff = gamma_n / (spin.muB * spin.meVtoGHzConversion)
spin.gfactorvector = [ge, g_n_eff]

# Hyperfine interaction
A_perp = 2.0e-3  # GHz
A_parallel = 8.0e-3  # GHz
spin.Avector[0] = np.array([A_perp, A_perp, A_parallel]) * spin.GHztomeVConversion

hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)
hamil.calc_EigStates(spin)

f_e = ge * spin.muB * B0 * spin.meVtoGHzConversion  # GHz
spin.f_RF = f_e  # use the electron carrier freuqency as the drive frequency
print(f"B = {spin.B} T")
print(f"Bare electron carrier: {f_e:.6f} GHz")


## Secular model and nuclear frequencies

`RotFrameSecular` retains hyperfine terms commuting with the electron
spin along the field. With $z'\parallel\mathbf B_0$, the conditional
nuclear Hamiltonian for $m_S=\pm1/2$ is

$$
H_n(m_S)/h=(-\nu_N+m_S A_{\mathrm{eff}})I_{z'}
                  +m_S B_{\mathrm{hf}} I_{x'},
$$

$$
A_{\mathrm{eff}}=A_\parallel\cos^2\theta+A_\perp\sin^2\theta,\qquad
|B_{\mathrm{hf}}|=|A_\parallel-A_\perp|\,|\sin\theta\cos\theta|.
$$

Here $A_{\mathrm{eff}}$ and $B_{\mathrm{hf}}$ denote frequency coefficients,
not magnetic fields. The transverse axis can be chosen so that
$B_{\mathrm{hf}}\geq0$. Define

$$
\nu_\pm=\sqrt{(-\nu_N\pm A_{\mathrm{eff}}/2)^2+(B_{\mathrm{hf}}/2)^2},
\qquad
k=\left(\frac{\nu_N B_{\mathrm{hf}}}{\nu_+\nu_-}\right)^2.
$$

The labels $\pm$ refer to electron $m_S$, not frequency ordering.
An aligned field or isotropic hyperfine tensor gives $B_{\mathrm{hf}}=0$
and no ESEEM in the ideal-pulse model.

See [Probst et al., Sections 2.2–2.3](https://mr.copernicus.org/articles/1/315/2020/)
for the secular model and Hahn envelope; their angular-frequency and
Zeeman conventions are converted here to ordinary frequencies and
Spinfinity's negative Zeeman sign.


In [ ]:
nu_N = gamma_n * B0  # GHz
A_eff = A_parallel * np.cos(theta) ** 2 + A_perp * np.sin(theta) ** 2
B_hf = abs((A_parallel - A_perp) * np.sin(theta) * np.cos(theta))
nu_plus = np.hypot(-nu_N + A_eff / 2, B_hf / 2)
nu_minus = np.hypot(-nu_N - A_eff / 2, B_hf / 2)
k = (nu_N * B_hf / (nu_plus * nu_minus)) ** 2

print(f"Nuclear Larmor frequency: {nu_N * 1e3:.6f} MHz")
print(f"A_eff = {A_eff * 1e3:.3f} MHz, |B_hf| = {B_hf * 1e3:.3f} MHz")
print(f"nu_plus = {nu_plus * 1e3:.6f} MHz")
print(f"nu_minus = {nu_minus * 1e3:.6f} MHz")
print(f"Modulation parameter k = {k:.6f}")

## Initial state and readout

We prepare a fully polarized electron along $\mathbf B_0$ and an unpolarized
nucleus: $\rho_0=\rho_e\otimes\mathbb1_n/2$. This does not correspond to thermal equilibrium.

For now we set `collapse_ops=[]` explicitly. Bath temperature and tunneling settings
are unused here; no electron or nuclear $T_1$ or $T_\phi$ is assumed.

For readout we use the `Sz_rot` operator, not laboratory `Sz[0]`, because the field is tilted.
The extra final $+x$ $\pi/2$ pulse of the function `CC.calc_HahnEcho_pluspihalf_Parallel` converts echo coherence to population.


In [ ]:
# Initial state
rho_e = np.array([[1 + n[2], n[0] - 1j * n[1]], [n[0] + 1j * n[1], 1 - n[2]]]) / 2
spin.psi0 = Qobj(np.kron(rho_e, np.eye(2) / 2))

# Collapse Operators
spin.collapse_ops = []

rabi_nominal = 0.100  # GHz = 100 MHz
spin.RabiRate = rabi_nominal
CC.RotFrameSecular(spin, spin.f_RF)
readout = [Qobj(spin.Sz_rot)]
I_field = Qobj(n[0] * spin.Sx[1] + n[1] * spin.Sy[1] + n[2] * spin.Sz[1])

print(f"Initial electron polarization: {2 * expect(readout[0], spin.psi0):.6f}")
print(f"Initial nuclear polarization: {2 * expect(I_field, spin.psi0):.6f}")
print(f"Nominal pi/2 duration: {1 / (4 * spin.RabiRate):.3f} ns")

## Ideal-pulse reference envelopes

Without relaxation, the normalized Hahn and three-pulse envelopes are

$$
V_2(\tau)=1-2k\sin^2(\pi\nu_+\tau)\sin^2(\pi\nu_-\tau),
$$

$$
\begin{aligned}
V_3(\tau,T)=1-\frac{k}{4}\bigl[
 &(1-\cos(2\pi\nu_-\tau))(1-\cos(2\pi\nu_+(T+\tau)))\
+&(1-\cos(2\pi\nu_+\tau))(1-\cos(2\pi\nu_-(T+\tau)))
\bigr].
\end{aligned}
$$

Here $\tau$ is **each** Hahn free delay. In three-pulse ESEEM, $T$ is the
scanned storage delay, while $\tau$ is fixed.

The three-pulse expression is given in
[Gunnar Jeschke's ESEEM notes](https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Electron_Paramagnetic_Resonance_(Jenschke)/08:_Measurement_of_Small_Hyperfine_Couplings/8.02:_ESEEM_and_HYSCORE).
Both references assume instantaneous pulses. Spinfinity also evolves the
drift Hamiltonian during its finite pulses.


In [ ]:
def hahn_envelope(tau, nu_plus, nu_minus, k):
    """Return the ideal normalized Hahn ESEEM envelope.

    Parameters
    ----------
    tau : array_like
        Each free delay in ns.
    nu_plus, nu_minus : float
        Conditional nuclear frequencies in GHz.
    k : float
        Dimensionless modulation parameter.

    Returns
    -------
    envelope : numpy.ndarray or float
        Dimensionless envelope with the shape of tau.

    Notes
    -----
    Assumes an unpolarized nucleus, instantaneous electron pulses,
    electron-secular free evolution, and no relaxation.
    """
    tau = np.asarray(tau, dtype=float)
    return (
        1
        - 2 * k * (np.sin(np.pi * nu_plus * tau) * np.sin(np.pi * nu_minus * tau)) ** 2
    )


def stimulated_envelope(T, tau, nu_plus, nu_minus, k):
    """Return the ideal normalized three-pulse ESEEM envelope.

    Parameters
    ----------
    T : array_like
        Storage delays in ns.
    tau : float
        Fixed free delay before and after storage, in ns.
    nu_plus, nu_minus : float
        Conditional nuclear frequencies in GHz.
    k : float
        Dimensionless modulation parameter.

    Returns
    -------
    envelope : numpy.ndarray or float
        Dimensionless envelope with the shape of T.

    Notes
    -----
    Assumes an unpolarized nucleus, instantaneous electron pulses,
    selection of the stimulated rephasing echo, and no relaxation.
    """
    T = np.asarray(T, dtype=float)
    return 1 - k / 4 * (
        (1 - np.cos(2 * np.pi * nu_minus * tau))
        * (1 - np.cos(2 * np.pi * nu_plus * (T + tau)))
        + (1 - np.cos(2 * np.pi * nu_plus * tau))
        * (1 - np.cos(2 * np.pi * nu_minus * (T + tau)))
    )

## Hahn echo and its Fourier spectrum

Simulate $\pi/2_x-\tau-\pi_x-\tau-\pi/2_x$ using $100 ~\mathrm{MHz}$ pulses.
The ideal spectrum versus $\tau$ contains $\nu_+$, $\nu_-$,
$|\nu_--\nu_+|$, and $\nu_-+\nu_+$. Finite pulses can alter amplitudes
and leave additional contributions.

Use `hf.calc_fft(times=..., signal=...)` with a Hann window and mean
subtraction. The times in $\mathrm{ns}$ give frequencies in $\mathrm{GHz}$, converted to $\mathrm{MHz}$ below.
The Fourier coordinate is $\tau$, **not $2\tau$**. The bin spacing is
$1/(N\Delta\tau)$.


In [ ]:
tau_array = np.arange(0.0, 10_000.0, 5.0)  # ns
_, hahn_result = CC.calc_HahnEcho_pluspihalf_Parallel(
    spin, tau_array=tau_array, e_ops=readout, n_jobs=1
)
hahn_simulated = 2 * hahn_result[:, 0]
hahn_ideal = hahn_envelope(tau_array, nu_plus, nu_minus, k)

# Calculate the Fourier transform
freq_hahn, fft_hahn = hf.calc_fft(times=tau_array, signal=hahn_simulated, window="hann")
_, fft_hahn_ideal = hf.calc_fft(times=tau_array, signal=hahn_ideal, window="hann")

# Expected frequencies 
hahn_lines = np.array([nu_plus, abs(nu_minus - nu_plus), nu_minus, nu_minus + nu_plus])
hahn_labels = [r"$\nu_+$", r"$|\nu_- - \nu_+|$", r"$\nu_-$", r"$\nu_- + \nu_+$"]


# Plots
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(tau_array / 1e3, hahn_simulated, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")
axes[0].plot(tau_array / 1e3, hahn_ideal, "--", label="Ideal")
axes[0].set(xlim=(0, 2), xlabel=r"$\tau$ ($\mu$s)", ylabel="Normalized Hahn signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_hahn * 1e3, abs(fft_hahn), label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")
axes[1].plot(freq_hahn * 1e3, abs(fft_hahn_ideal), "--", label="Ideal")
for frequency, label in zip(hahn_lines, hahn_labels, strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, alpha=0.6)
    axes[1].text(
        frequency * 1e3,
        0.98,
        label,
        rotation=90,
        va="top",
        ha="right",
        transform=axes[1].get_xaxis_transform(),
    )
axes[1].set(
    xlim=(0, 14),
    xlabel="Frequency conjugate to tau (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print(f"FFT bin spacing: {(freq_hahn[1] - freq_hahn[0]) * 1e3:.3f} MHz")

## Eight-step three-pulse ESEEM

Simulate $\pi/2-\tau-\pi/2-T-\pi/2-\tau-\pi/2$ using
`calc_3pulseESEEM_pluspihalf_PhaseCycled8`.
Its eight signed measurements are averaged, selecting the rephasing
stimulated contribution in the ideal-pulse limit. The final pulse is
population readout, not an extra free-evolution interval.

Choose $\tau=1/(2\nu_-)$ to maximize the ideal $\nu_+$ coefficient.
Only $\nu_+$ and $\nu_-$ occur in the ideal $T$ spectrum.
The exact package cycle is documented in the function docstring.


In [ ]:
T_array = np.arange(0.0, 10_000.0, 5.0)  # ns
tau_bright = 1 / (2 * nu_minus)

_, stimulated_result = CC.calc_3pulseESEEM_pluspihalf_PhaseCycled8(
    spin, T_array=T_array, tau=tau_bright, e_ops=readout, n_jobs=1
)
stimulated_simulated = 4 * stimulated_result[:, 0]
stimulated_ideal = stimulated_envelope(T_array, tau_bright, nu_plus, nu_minus, k)

# Fourrier
freq_stimulated, fft_stimulated = hf.calc_fft(
    times=T_array, signal=stimulated_simulated, window="hann"
)
_, fft_stimulated_ideal = hf.calc_fft(
    times=T_array, signal=stimulated_ideal, window="hann"
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(T_array / 1e3, stimulated_simulated, label="Spinfinity, 100 MHz")
axes[0].plot(T_array / 1e3, stimulated_ideal, "--", label="Instantaneous pulses")
axes[0].set(xlim=(0, 2), xlabel=r"$T$ ($\mu$s)", ylabel="Normalized stimulated signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_stimulated * 1e3, abs(fft_stimulated), label="Spinfinity")
axes[1].plot(freq_stimulated * 1e3, abs(fft_stimulated_ideal), "--", label="Ideal")
for frequency, label in zip([nu_plus, nu_minus], [r"$\nu_+$", r"$\nu_-$"], strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, label=label)
axes[1].set(
    xlim=(0, 12),
    xlabel="Frequency conjugate to T (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print(f"Fixed tau = {tau_bright:.6f} ns")

## Thermal initialization and the effect of temperature
A more realistic experimental setting is that the state populations are in thermal equilibrium.
We discuss this effect on the Hahn echo experiment. The thermal state occupation of the electron spin modifies the amplitudes of the oscillations and Fourier peaks. 
A polarization of the nuclear spin has a similar effect. The nuclear frequencies stay the same.

In [ ]:
# Initialize thermal
spin.T = 0.2
lind.initialize_thermal(spin)
# spin.psi0=Qobj([1,0,0,0])

tau_array = np.arange(0.0, 10_000.0, 5.0)  # ns
_, hahn_result = CC.calc_HahnEcho_pluspihalf_Parallel(
    spin, tau_array=tau_array, e_ops=readout, n_jobs=1
)
hahn_simulated = 2 * hahn_result[:, 0]
hahn_ideal = hahn_envelope(tau_array, nu_plus, nu_minus, k)

# Calculate the Fourier transform
freq_hahn, fft_hahn = hf.calc_fft(times=tau_array, signal=hahn_simulated, window="hann")
_, fft_hahn_ideal = hf.calc_fft(times=tau_array, signal=hahn_ideal, window="hann")

# Expected frequencies 
hahn_lines = np.array([nu_plus, abs(nu_minus - nu_plus), nu_minus, nu_minus + nu_plus])
hahn_labels = [r"$\nu_+$", r"$|\nu_- - \nu_+|$", r"$\nu_-$", r"$\nu_- + \nu_+$"]


# Plots
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(tau_array / 1e3, hahn_simulated, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz, T={spin.T:.2f} K")
axes[0].plot(tau_array / 1e3, hahn_ideal, "--", label="Ideal")
axes[0].set(xlim=(0, 2), xlabel=r"$\tau$ ($\mu$s)", ylabel="Normalized Hahn signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_hahn * 1e3, abs(fft_hahn), label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz, T={spin.T:.2f} K")
axes[1].plot(freq_hahn * 1e3, abs(fft_hahn_ideal), "--", label="Ideal")
for frequency, label in zip(hahn_lines, hahn_labels, strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, alpha=0.6)
    axes[1].text(
        frequency * 1e3,
        0.98,
        label,
        rotation=90,
        va="top",
        ha="right",
        transform=axes[1].get_xaxis_transform(),
    )
axes[1].set(
    xlim=(0, 14),
    xlabel="Frequency conjugate to tau (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Weaker drive
The ideal analytical solution holds for infinite sharp pulses. For finite pulses the signal changes, this becomes problematic when the Rabi rate becomes compareable to the nuclear frequencies we try to observe. 

In [ ]:
# Different driving strengths
rates_MHz = [1.0, 100.0]
rates_GHz = np.asarray(rates_MHz) * 1.0e-3

# Initial state
rho_e = np.array([[1 + n[2], n[0] - 1j * n[1]], [n[0] + 1j * n[1], 1 - n[2]]]) / 2
spin.psi0 = Qobj(np.kron(rho_e, np.eye(2) / 2))

# Ideal solution
tau_array = np.arange(0.0, 10_000.0, 5.0)  # ns
hahn_ideal = hahn_envelope(tau_array, nu_plus, nu_minus, k)
_, fft_hahn_ideal = hf.calc_fft(times=tau_array, signal=hahn_ideal, window="hann")

# simulate the different rates
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
for i, rate in enumerate(rates_GHz):
    spin.RabiRate =  rate
    _, hahn_result = CC.calc_HahnEcho_pluspihalf_Parallel(
        spin, tau_array=tau_array, e_ops=readout, n_jobs=1
    )
    hahn_simulated = 2 * hahn_result[:, 0]
    freq_hahn, fft_hahn = hf.calc_fft(times=tau_array, signal=hahn_simulated, window="hann")

    # Add to plot
    axes[0].plot(tau_array / 1e3, hahn_simulated, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")
    axes[1].plot(freq_hahn * 1e3, abs(fft_hahn), label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")

# Remaining plot
axes[0].plot(tau_array / 1e3, hahn_ideal, "--", label="Ideal")
axes[0].set(xlim=(0, 2), xlabel=r"$\tau$ ($\mu$s)", ylabel="Normalized Hahn signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_hahn * 1e3, abs(fft_hahn_ideal), "--", label="Ideal")
for frequency, label in zip(hahn_lines, hahn_labels, strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, alpha=0.6)
    axes[1].text(
        frequency * 1e3,
        0.98,
        label,
        rotation=90,
        va="top",
        ha="right",
        transform=axes[1].get_xaxis_transform(),
    )
axes[1].set(
    xlim=(0, 14),
    xlabel="Frequency conjugate to tau (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

# Restore Rabi rate
spin.RabiRate = 0.1 # GHz

## Dephasing and relaxation 

### Building collapse operators

We construct the dephasing and relaxation operators for the electron spin. The nuclear spin can usually be neglected since it's lifetime and coherence is typically much longer

In [ ]:
# Tunneling Parameters
spin.T=0.2
spin.V_DC = 0.0 # V
spin.U = [0, 0]
spin.G_st = 2e-11 #S
spin.G_tt = 1
spin.G_ss = [1e-10, 0] #S

bath.calc_Rates(spin)

## Collapse Operators
# Calculate T1 events from tunneling rates
c_ops_T1 = lind.calc_collapseOperators(spin, threshold_percent=None, recalcRates=True)
T_phi_e = 1000  # ns

c_ops_Tphi_e = lind.calc_dephasingOperator(spin, spin_idx=0, T_phi=T_phi_e)

spin.collapse_ops = c_ops_T1 + [c_ops_Tphi_e] 

### Hahn echo simulation

Through the relaxation and dephasing of the electron spin, the signal decays.

In [ ]:
# Initial state
rho_e = np.array([[1 + n[2], n[0] - 1j * n[1]], [n[0] + 1j * n[1], 1 - n[2]]]) / 2
spin.psi0 = Qobj(np.kron(rho_e, np.eye(2) / 2))

tau_array = np.arange(0.0, 10_000.0, 5.0)  # ns
_, hahn_result = CC.calc_HahnEcho_pluspihalf_Parallel(
    spin, tau_array=tau_array, e_ops=readout, n_jobs=1
)
hahn_simulated = 2 * hahn_result[:, 0]
hahn_ideal = hahn_envelope(tau_array, nu_plus, nu_minus, k)

# Subtract exponential background
x_fit, y_fit, popt, perr = hf.fit_exponential(tau_array, hahn_simulated)
hahn_processed = hahn_simulated - y_fit


# Calculate the Fourier transform
freq_hahn, fft_hahn = hf.calc_fft(times=tau_array, signal=hahn_simulated, window="hann")
_, fft_hahn_processed = hf.calc_fft(times=tau_array, signal=hahn_processed, window="hann")
_, fft_hahn_ideal = hf.calc_fft(times=tau_array, signal=hahn_ideal, window="hann")

# Expected frequencies 
hahn_lines = np.array([nu_plus, abs(nu_minus - nu_plus), nu_minus, nu_minus + nu_plus])
hahn_labels = [r"$\nu_+$", r"$|\nu_- - \nu_+|$", r"$\nu_-$", r"$\nu_- + \nu_+$"]


# Plots
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(tau_array / 1e3, hahn_simulated, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")
axes[0].plot(tau_array / 1e3, hahn_processed + 0.5, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz, processed")
axes[0].plot(tau_array / 1e3, hahn_ideal, "--", label="Ideal")
axes[0].plot(x_fit / 1e3, y_fit, "--", linewidth=0.2, color='k', label="Exp fit")
axes[0].set(xlim=(0, 2), xlabel=r"$\tau$ ($\mu$s)", ylabel="Normalized Hahn signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_hahn * 1e3, abs(fft_hahn), label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz")
axes[1].plot(freq_hahn * 1e3, abs(fft_hahn_processed)*50, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz, processed x50")
axes[1].plot(freq_hahn * 1e3, abs(fft_hahn_ideal), "--", label="Ideal")
for frequency, label in zip(hahn_lines, hahn_labels, strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, alpha=0.6)
    axes[1].text(
        frequency * 1e3,
        0.98,
        label,
        rotation=90,
        va="top",
        ha="right",
        transform=axes[1].get_xaxis_transform(),
    )
axes[1].set(
    xlim=(0, 14),
    xlabel="Frequency conjugate to tau (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print(f"Hahn echo decay: {popt[1]:.0f} ns")

### three-pulse ESEEM
Also for the three-pulse ESEEM the relaxation and decoherence influences the signal. However, not that drastically

In [ ]:
T_array = np.arange(0.0, 10_000.0, 5.0)  # ns
tau_bright = 1 / (2 * nu_minus)

_, stimulated_result = CC.calc_3pulseESEEM_pluspihalf_PhaseCycled8(
    spin, T_array=T_array, tau=tau_bright, e_ops=readout, n_jobs=1
)
stimulated_simulated = 4 * stimulated_result[:, 0]
stimulated_ideal = stimulated_envelope(T_array, tau_bright, nu_plus, nu_minus, k)

# Subtract exponential background
x_fit, y_fit, popt, perr = hf.fit_exponential(tau_array, stimulated_simulated)
stimulated_processed = stimulated_simulated - y_fit

# Fourrier
freq_stimulated, fft_stimulated = hf.calc_fft(
    times=T_array, signal=stimulated_simulated, window="hann"
)
_, fft_stimulated_processed = hf.calc_fft(
    times=T_array, signal=stimulated_processed, window="hann"
)
_, fft_stimulated_ideal = hf.calc_fft(
    times=T_array, signal=stimulated_ideal, window="hann"
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(T_array / 1e3, stimulated_simulated, label="Spinfinity, 100 MHz")
axes[0].plot(T_array / 1e3, stimulated_processed + 0.5, label=f"Spinfinity, {spin.RabiRate*1e3:.0f} MHz, processed")
axes[0].plot(T_array / 1e3, stimulated_ideal, "--", label="ideal")
axes[0].plot(x_fit / 1e3, y_fit, "--", linewidth=0.2, color='k', label="Exp fit")
axes[0].set(xlim=(0, 2), xlabel=r"$T$ ($\mu$s)", ylabel="Normalized stimulated signal")
axes[0].legend(fontsize=8)

axes[1].plot(freq_stimulated * 1e3, abs(fft_stimulated), label="Spinfinity")
axes[1].plot(freq_stimulated * 1e3, abs(fft_stimulated_processed)*5, label="Spinfinity processec, x5")
axes[1].plot(freq_stimulated * 1e3, abs(fft_stimulated_ideal), "--", label="Ideal")
for frequency, label in zip([nu_plus, nu_minus], [r"$\nu_+$", r"$\nu_-$"], strict=True):
    axes[1].axvline(frequency * 1e3, color="gray", lw=0.8, label=label)
axes[1].set(
    xlim=(0, 12),
    xlabel="Frequency conjugate to T (MHz)",
    ylabel="One-sided FFT magnitude",
)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print(f"ESEEM decay: {popt[1]:.0f} ns")

## Background literature

1. [Mims (1972), *Envelope Modulation in Spin-Echo Experiments*,
   Physical Review B 5, 2409](https://doi.org/10.1103/PhysRevB.5.2409).
   General theoretical foundation.
2. [Schweiger and Jeschke (2001), *Principles of Pulse Electron Paramagnetic
   Resonance*, Chapter 10](https://doi.org/10.1093/oso/9780198506348.003.0010).
   Standard pulse-EPR treatment.
3. [Gunnar Jeschke, *ESEEM and HYSCORE*, three-pulse ESEEM section](https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Electron_Paramagnetic_Resonance_(Jenschke)/08:_Measurement_of_Small_Hyperfine_Couplings/8.02:_ESEEM_and_HYSCORE).
   Open derivation of the conditional three-pulse envelopes.
4. [Probst et al. (2020), *Hyperfine spectroscopy in a quantum-limited
   spectrometer*, Sections 2.2–2.3](https://doi.org/10.5194/mr-1-315-2020).
   Accessible spin-1/2 theory and experimental context. The illustrative
   carbon parameters here are not taken from their experimental systems.
